## FREE UNCENSURED AI - **Pony diffusion V6 XL** (PHOTO)
Author GitHub - [**ArteM1421**](https://github.com/ArteM1421)  
GitHub REPO - [**CLICK**](https://github.com/ArteM1421/FREE-UNCENSURED-AI)


⚠️ **I am not responsible for what you have done in this tool. All responsibility lies with you.**

In [ ]:
#@title **🚀 STEP 1: INSTALL REQUIREMENTS**
!pip install -U diffusers transformers accelerate

In [ ]:
#@title **🚀 STEP 2: START AI**

import os
import gc
import time
from datetime import datetime
from pathlib import Path

import torch
from diffusers import StableDiffusionXLPipeline
from IPython.display import display, clear_output

# ----------------------------- CONFIG -----------------------------
MODEL_ID   = "stablediffusionapi/Pony-Diffusion-V6-XL"
OUTPUT_DIR = Path("generated")
OUTPUT_DIR.mkdir(exist_ok=True)

NEGATIVE_PROMPT = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry"

# ----------------------------- MENU -------------------------------
MENU = r"""
╔══════════════════════════════════════════════════════════════╗
║                                                              ║
║        ███████╗██████╗ ███████╗███████╗                       ║
║        ██╔════╝██╔══██╗██╔════╝██╔════╝                       ║
║        █████╗  ██████╔╝█████╗  █████╗                         ║
║        ██╔══╝  ██╔══██╗██╔══╝  ██╔══╝                         ║
║        ██║     ██║  ██║███████╗███████╗                       ║
║        ╚═╝     ╚═╝  ╚═╝╚══════╝╚══════╝                       ║
║                                                              ║
║           U N C E N S O R E D   A I                          ║
║                                                              ║
║                  GITHUB: ARTEM1421                           ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝

  [1]  Generate an image
  [2]  Generate multiple images
  [3]  Settings
  [0]  Exit
"""

# ----------------------------- SETTINGS ---------------------------
settings = {
    "steps": 30,
    "guidance": 7.0,
    "width": 832,
    "height": 1216,
    "seed": -1,          # -1 = random
}

# ----------------------------- PIPELINE ---------------------------
_pipe = None

def get_pipe():
    """Lazy-load the pipeline — model loads only on first generation."""
    global _pipe
    if _pipe is None:
        print("\n[!] Loading model, please wait (first time ~1-2 min)...\n")
        _pipe = StableDiffusionXLPipeline.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            variant="fp16",
            device_map="cuda",
        )
        try:
            _pipe.enable_vae_slicing()
            _pipe.enable_vae_tiling()
        except Exception:
            pass
        _pipe.set_progress_bar_config(disable=False)
        print("\n[✓] Model loaded.\n")
    return _pipe

# ----------------------------- GENERATE ---------------------------
def generate(prompt: str, n: int = 1):
    pipe = get_pipe()

    seed = settings["seed"]
    if seed == -1:
        seed = torch.randint(0, 2**32 - 1, (1,)).item()
    gen = torch.Generator(device="cuda").manual_seed(seed)

    print(f"\n» Prompt : {prompt}")
    print(f"» Seed   : {seed}")
    print(f"» Size   : {settings['width']}x{settings['height']}")
    print(f"» Steps  : {settings['steps']}  |  CFG: {settings['guidance']}\n")

    start = time.time()
    result = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=settings["steps"],
        guidance_scale=settings["guidance"],
        width=settings["width"],
        height=settings["height"],
        generator=gen,
        num_images_per_prompt=n,
    )
    elapsed = time.time() - start

    saved = []
    for i, img in enumerate(result.images):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = OUTPUT_DIR / f"{ts}_{i}_seed{seed}.png"
        img.save(path)
        saved.append(path)
        display(img)

    print(f"\n[✓] Done in {elapsed:.1f}s. Saved to: {OUTPUT_DIR}/")
    for p in saved:
        print(f"    • {p.name}")

# ----------------------------- DISPLAY ----------------------------
def print_menu():
    clear_output(wait=True)
    print(MENU)
    print(f"  Current settings: "
          f"{settings['width']}x{settings['height']}, "
          f"{settings['steps']} steps, CFG {settings['guidance']}, "
          f"seed {settings['seed']}\n")

def settings_menu():
    print("\n--- SETTINGS ---")
    try:
        s = input(f"Steps [{settings['steps']}]: ").strip()
        if s: settings["steps"] = int(s)

        g = input(f"CFG [{settings['guidance']}]: ").strip()
        if g: settings["guidance"] = float(g)

        w = input(f"Width [{settings['width']}]: ").strip()
        if w: settings["width"] = int(w)

        h = input(f"Height [{settings['height']}]: ").strip()
        if h: settings["height"] = int(h)

        sd = input(f"Seed (-1=random) [{settings['seed']}]: ").strip()
        if sd: settings["seed"] = int(sd)
    except ValueError:
        print("[!] Invalid value, settings unchanged.")
        return
    print("[✓] Settings updated.")

# ----------------------------- MAIN LOOP --------------------------
def main():
    while True:
        print_menu()
        choice = input("Choice: ").strip()

        if choice == "0":
            print("\nGoodbye! 👋")
            break

        elif choice == "1":
            prompt = input("\nEnter prompt: ").strip()
            if not prompt:
                print("[!] Empty prompt.")
                continue
            try:
                generate(prompt, n=1)
            except Exception as e:
                print(f"[✗] Error: {e}")

        elif choice == "2":
            prompt = input("\nEnter prompt: ").strip()
            if not prompt:
                print("[!] Empty prompt.")
                continue
            try:
                n = int(input("How many images? [4]: ").strip() or 4)
                generate(prompt, n=n)
            except Exception as e:
                print(f"[✗] Error: {e}")

        elif choice == "3":
            settings_menu()

        else:
            print("[!] Unknown command.")

        input("\nPress Enter to continue...")


if __name__ == "__main__":
    main()